# Streamlit


## **Información general**


**streamlit_app.py**  Está dividido en 5 partes:

| Bloque              | Qué hace                                                                 |
|--------------------|-------------------------------------------------------------------------|
| 1. CARGA DE DATOS  | Lee `src/data/df_clean.csv` con `@st.cache_data`                         |
| 2. CARGA DEL MODELO| Carga los 3 `.pkl` con `@st.cache_resource`                              |
| 3. PREDICCIÓN      | Función `predecir_hit()`                               |
| 4. GRÁFICOS        | Las 3 funciones de visualización                                         |
| 5. APP STREAMLIT   | La interfaz, una sola página                                             |

* Función **predecir_hit**:

1. **Codificación de género**  
   Convierte el género de texto a número usando `encoder_tag.pkl` (ej: `'pop' → 37`).

2. **Estimación de playcount**  
   Calcula los plays estimados: `oyentes × engagement`.

3. **Determinación de hit**  
   Si los plays estimados superan los 5 millones → `is_hit = 1`.  
   Si no → `is_hit = 0`.

4. **Construcción de features**  
   Crea la fila con las 9 columnas en el orden exacto que espera el modelo.

5. **Predicción**  
   Llama al modelo con `predict_proba()` y multiplica por 100 para obtener el porcentaje.

6. **Extra (importante)**  
   `reindex(columns=features, fill_value=0)` garantiza que las columnas estén en el mismo orden que en el entrenamiento, independientemente de cómo se definan en el código.


## **Explicación de los bloques del archivo**

### **BLOQUE 1: CARGA DE DATOS**

Lee el archivo df_clean.csv que ya tiene todas las columnas calculadas del notebook (duration_min, is_hit, tag, playcount, etc.). 

El @st.cache_data es un decorador de Streamlit que hace que el archivo se lea una sola vez — si el usuario interactúa con la app (mueve un slider, pulsa un botón), Streamlit no vuelve a leer el CSV desde disco, usa el que ya tiene en memoria. 

**¿Por qué `@st.cache_data`?** 
Sin esto la app sería lenta porque releería 34.000 filas en cada clic. Streamlit re-ejecuta todo el archivo desde el principio en **cada interacción del usuario** (clic en un botón, mover un slider, cambiar un selectbox). Sin caché, leería el CSV de 34.000 filas en cada clic. Con caché, lo lee una vez y reutiliza el resultado durante toda la sesión.

**¿Por qué `df_clean.csv` y no reconstruir desde los CSVs crudos?**  
- El dataset ya tiene todas las columnas calculadas (`log_listeners`, `is_hit`, `tag`, etc.)
- Evita repetir toda la lógica de limpieza y feature engineering en producción
- Más rápido de cargar — un archivo vs tres más procesado

**¿Qué es `errors="coerce"` en `pd.to_numeric()`?**  
Convierte los valores que no se pueden parsear a número en `NaN` en lugar de lanzar un error. Si la columna tiene strings mezclados con números, no rompe la ejecución.

###  **BLOQUE 2: CARGA DEL MODELO**

Carga tres archivos que se generaron en el notebook:
**Los tres artefactos del modelo:**

| **Archivo** | **Qué contiene** | **Para qué** |
|---|---|---|
| `modelo_hit.pkl` | RandomForestClassifier entrenado | Hace las predicciones |
| `encoder_tag.pkl` | LabelEncoder con 313 géneros | Convierte `'pop' → 37`, `'rock' → 42` |
| `features.pkl` | Lista de 9 columnas en orden | Garantiza el orden correcto de inputs |

**`@st.cache_resource` vs `@st.cache_data`:**  
Los modelos de sklearn son **objetos grandes** que no deben copiarse. `@st.cache_resource` los comparte entre sesiones de usuario sin copiarlos. `@st.cache_data` crearía una copia por sesión, desperdiciando memoria RAM.

**¿Por qué `joblib` y no `pickle`?**  
`joblib` es más eficiente con arrays numpy grandes (como los árboles internos del Random Forest). Genera archivos más pequeños y carga más rápido. `pickle` es más lento y genera archivos de mayor tamaño para este tipo de objetos.


###  **BLOQUE 3: PREDICCIÓN**

**Paso 1:** El modelo no entiende texto, el encoder convierte 'pop' en 37. Si el usuario escribe un género que el modelo nunca vio durante el entrenamiento, se pone 0 como valor por defecto.

**Paso 2:** Para una canción nueva no sabes cuántas reproducciones tendrá, pero el modelo las necesita porque fue entrenado con esa columna. La estimación es: oyentes × cuántas veces escucha cada uno = plays totales estimados. Si ese número supera los 5 millones (umbral p90 del dataset), is_hit_est = 1.

**Paso 3:** Se construye una tabla de una sola fila con todas las columnas. El .reindex(columns=features, fill_value=0) reordena las columnas en el orden exacto de features.pkl — si falta alguna columna la rellena con 0.

**Paso 4:** predict_proba() devuelve dos números: la probabilidad de que sea 0 (no hit) y la probabilidad de que sea 1 (hit). El [0][1] coge el segundo, que es el que nos interesa. Se multiplica por 100 para tenerlo en porcentaje.

**Clasificación del resultado:**

| **Probabilidad** | **Emoji** | **Etiqueta** |
|---|---|---|
| ≥ 70% | 🚀 | Hit potencial |
| 45% - 70% | 🟡 | Potencial medio |
| < 45% | 📉 | Bajo potencial |

###  **BLOQUE 4: GRÁFICOS**

Tres funciones, todas con el mismo patrón: reciben el DataFrame, crean una figura con matplotlib y la devuelven sin mostrarla. Streamlit la muestra después con st.pyplot().

**¿Por qué `return fig` y no `plt.show()`?**  
`plt.show()` abre una ventana emergente del sistema operativo — no funciona en un servidor web. Streamlit necesita recibir el objeto figura y renderizarlo él mismo dentro de la interfaz con `st.pyplot(fig)`.

**Las tres visualizaciones:**

| **Función** | **Qué muestra** | **Fuente de datos** |
|---|---|---|
| `plot_top_tracks(df, n=15)` | Top 15 canciones por reproducciones | `df['playcount']` |
| `plot_genres(df, n=12)` | Top géneros por reproducciones totales | `df.groupby('tag')['playcount'].sum()` |
| `plot_heatmap(df)` | Reproducciones medias por Género × País | Solo países reales (excluye GLOBAL y UNKNOWN) |


###  **BLOQUE 5: APP STREAMLIT**


**¿Cómo funciona Streamlit?**  
Streamlit re-ejecuta todo el archivo Python de **arriba a abajo** en cada interacción. Por eso el caché es crítico — sin él, cada clic recargaría el CSV y el modelo.

**`st.columns([1, 1])`** — divide la pantalla en dos columnas iguales. Los inputs del predictor van a la izquierda, el resultado a la derecha.

**`st.tabs(["Tab 1", "Tab 2", "Tab 3"])`** — organiza los tres gráficos en pestañas. Evita que la página sea infinitamente larga.

**`if predecir:`** — solo ejecuta la predicción cuando el usuario pulsa el botón. Sin este condicional, la app intentaría predecir al cargar, antes de que el usuario introduzca parámetros.

## Errores durante el proceso

| **#** | **Error** | **Categoría** | 
|---|---|---|
| 1 | `ModuleNotFoundError: No module named 'joblib'` | `[ENV][DEPS]` |
| 2 | `FileNotFoundError: df_clean.csv` | `[ENV][CONFIG]` | 
| 3 | `NameError: clean_and_feature_engineer not defined` | `[NOTEBOOK]` | 
| 4 | Librerías pesadas bloquean el deploy | `[ENV][DEPS]` | 
| 5 | Deploy bloqueado por organización 4GeeksAcademy | `[VCS][AUTH]` |
| 6 | `git push rejected: File exceeds 100MB` | `[VCS]` | 
| 7 | `streamlit run streamlit_app.pyp` — typo | `[NOTEBOOK]` |
| 8 | App en Streamlit Cloud no encuentra el archivo | `[CONFIG]` |

> **ERROR 1:** ModuleNotFoundError: No module named 'joblib

**Etiquetas:** `[ENV]` `[DEPS]`


**Causa:**  
La librería no estaba instalada en el entorno, o el archivo `requirements.txt` no existía o tenía un typo en el nombre (`requirement.txt` sin `s`).

**Solución:** `requirements.txt` es el archivo que Streamlit Cloud lee para saber qué librerías instalar antes de arrancar la app. Si no existe, o tiene un typo, la app no arranca. 

> **ERROR 2:**  Rutas de archivos

**Dos partes de la solución:**

1. En el notebook — generar el archivo en la ruta correcta:
```python
df_clean.to_csv("../src/data/df_clean.csv", index=False)
```

2. En la app — usar rutas absolutas desde `__file__` para que funcione en cualquier entorno:
```python
_BASE     = os.path.dirname(os.path.abspath(__file__))
_DATA_CSV = os.path.join(_BASE, "src", "data", "df_clean.csv")
```

**¿Por qué rutas absolutas?**  
Las rutas relativas como `"src/data/df_clean.csv"` dependen de dónde se ejecuta el comando. Si lo ejecutas desde una carpeta diferente, el path falla. Las rutas absolutas construidas desde `__file__` (la ubicación del script) funcionan independientemente del directorio de trabajo.

> **ERROR 3:** Librerías pesadas bloquean el deploy

**Etiquetas:** `[ENV]` `[DEPS]`

**Síntoma:**  
La app tardaba más de 15 minutos en arrancar en Streamlit Cloud o fallaba el build por timeout.

**Causa:**  
`xgboost` y `statsmodels` en `requirements.txt` tienen muchas dependencias y tardan mucho en instalarse. Si no se usan en la app, no tienen que estar.

**Solución:**  Generar app desde local y eliminar xgboost y statsmodels de requirements.txt

> **ERROR 4:**  git push rejected: File exceeds 100MB
> ```
> remote: error: File src/models/modelo_plays_reg.pkl is 259.93 MB
> remote: error: GH001: Large files detected
> error: failed to push some refs to 'https://github.com/...'
> ```

**Causa:**  
El modelo de regresión (`rf_reg`) pesa 259MB — supera el límite de 100MB de GitHub. Se había hecho commit del archivo por error.

**Solución:** quitar el archivo del seguimiento de Git usando git rm --chached.

**¿Qué hace `git rm --cached`?**  
Elimina el archivo del seguimiento de Git (del índice) sin borrarlo del disco. El archivo sigue estando en local pero Git deja de rastrearlo.

## Ejecución de Streamlit en local


```bash
# 1. Instalar dependencias
pip install streamlit pandas numpy scikit-learn joblib matplotlib seaborn

# 2. Ir a la carpeta del proyecto
cd c-chico23_ProyectoFinal-main

# 3. Ejecutar
streamlit run streamlit_app.py

# 4. Abrir en el navegador
# → http://localhost:8501
```

**Ventajas para la presentación:**
- No depende de conexión a internet
- Arranque en segundos (sin cold start de la nube)
- Más estable durante la demo



## Tabla de decisiones técnicas



| **Decisión** | **Alternativa descartada** | **Por qué se eligió** |
|---|---|---|
| Un solo `streamlit_app.py` | Módulos separados (app.py, predict.py...) | Más simple de desplegar y depurar |
| `df_clean.csv` en producción | Reconstruir desde CSVs crudos | Evita repetir pipeline, más rápido |
| `joblib` para modelos | `pickle` | Más eficiente con arrays numpy |
| `@st.cache_data` para datos | Sin caché | Evita releer 34k filas en cada clic |
| `@st.cache_resource` para modelo | `@st.cache_data` | No copia objetos grandes en memoria |
| `.reindex(fill_value=0)` | Orden manual de columnas | Garantiza compatibilidad con el modelo |
| Rutas con `os.path.abspath(__file__)` | Rutas relativas simples | Funciona en cualquier entorno |
| Repo personal en GitHub | Repo de organización 4GeeksAcademy | Evita restricciones OAuth en Streamlit Cloud |